In [1]:
print("Hello, I am ready to build the COBOL translator")

Hello, I am ready to build the COBOL translator


In [4]:
!pip install unsloth


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.4/79.4 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 118.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 111.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 115.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="/content/drive/MyDrive/cobol_to_python_1000.jsonl",
    split="train"
)

print(f"Total examples: {len(dataset)}")
print(f"Columns: {dataset.column_names}")
print(f"First example: {dataset[0]}")

Generating train split: 0 examples [00:00, ? examples/s]

Total examples: 1000
Columns: ['conversations']
First example: {'conversations': [{'role': 'user', 'content': "Translate this COBOL paragraph to modern Python. Use the 'decimal' module for all currency and math. Include a pytest assertion.\n\n01 WS-BASE PIC S9(5)V99 VALUE 1994.83.\n01 WS-DEDUCT PIC S9(5)V99 VALUE 931.51.\nSUBTRACT WS-DEDUCT FROM WS-BASE."}, {'role': 'assistant', 'content': "from decimal import Decimal\n\nws_base = Decimal('1994.83')\nws_deduct = Decimal('931.51')\nws_base = ws_base - ws_deduct\n\ndef test_subtract_from():\n    assert ws_base == Decimal('1063.32')"}]}


In [3]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-1.5B-Instruct",
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)

print("Model loaded successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.5: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded successfully!


In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print("LoRA adapters added!")

Unsloth 2026.8.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


LoRA adapters added!


In [5]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="/content/drive/MyDrive/cobol_to_python_1000.jsonl",
    split="train"
)

def format_example(example):
    messages = example["conversations"]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

dataset = dataset.map(format_example)

print(f"Dataset formatted. Total examples: {len(dataset)}")
print(f"Sample:\n{dataset[0]['text'][:200]}...")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset formatted. Total examples: 1000
Sample:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Translate this COBOL paragraph to modern Python. Use the 'decimal' module for all cur...


In [6]:
from trl import SFTConfig

training_args = SFTConfig(
    dataset_text_field="text",
    max_seq_length=1024,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_ratio=0.1,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
    output_dir="/content/drive/MyDrive/cobol_training_outputs",
    save_steps=100,
    save_total_limit=2,
    report_to="none",
)

print("Training config set!")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training config set!


In [7]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
)

print("Starting training...")
trainer.train()
print("Training complete!")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 125
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,2.147336
20,0.958825
30,0.341791
40,0.238043
50,0.207231
60,0.193744
70,0.168276
80,0.157495
90,0.133923
100,0.151625


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/cobol_training_outputs/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/cobol_training_outputs/checkpoint-125/tokenizer_config.json.


Training complete!


In [8]:
model.save_pretrained("/content/drive/MyDrive/cobol_adapter")
tokenizer.save_pretrained("/content/drive/MyDrive/cobol_adapter")

print("Adapter saved to Google Drive!")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/cobol_adapter/tokenizer_config.json.


Adapter saved to Google Drive!


In [9]:
FastLanguageModel.for_inference(model)

test_prompt = """Translate this COBOL paragraph to modern Python. Use the 'decimal' module for all currency and math. Include a pytest assertion.

01 WS-PRICE PIC S9(5)V99 VALUE 250.00.
01 WS-TAX-RATE PIC V99 VALUE 0.08.
COMPUTE WS-TAX = WS-PRICE * WS-TAX-RATE."""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.1,
    top_p=0.9,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Translate this COBOL paragraph to modern Python. Use the 'decimal' module for all currency and math. Include a pytest assertion.

01 WS-PRICE PIC S9(5)V99 VALUE 250.00.
01 WS-TAX-RATE PIC V99 VALUE 0.08.
COMPUTE WS-TAX = WS-PRICE * WS-TAX-RATE. '''

from decimal import Decimal

ws_price = Decimal('250.00')
ws_tax_rate = Decimal('0.08')
ws_tax = ws_price * ws_tax_rate

def test_compute_tax():
    assert ws_tax == Decimal('20.0')


In [10]:
FastLanguageModel.for_inference(model)

test_prompt = """Translate this COBOL paragraph to modern Python. Use the 'decimal' module for all currency and math. Include a pytest assertion.

01 WS-AMOUNT PIC S9(7)V99 VALUE 1500.00.
01 WS-DISCOUNT PIC S9(3)V99 VALUE ZERO.
IF WS-AMOUNT > 1000.00
   COMPUTE WS-DISCOUNT = WS-AMOUNT * 0.10
ELSE
   IF WS-AMOUNT > 500.00
      COMPUTE WS-DISCOUNT = WS-AMOUNT * 0.05
   ELSE
      MOVE ZERO TO WS-DISCOUNT
   END-IF
END-IF."""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.1,
    top_p=0.9,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Translate this COBOL paragraph to modern Python. Use the 'decimal' module for all currency and math. Include a pytest assertion.

01 WS-AMOUNT PIC S9(7)V99 VALUE 1500.00.
01 WS-DISCOUNT PIC S9(3)V99 VALUE ZERO.
IF WS-AMOUNT > 1000.00
   COMPUTE WS-DISCOUNT = WS-AMOUNT * 0.10
ELSE
   IF WS-AMOUNT > 500.00
      COMPUTE WS-DISCOUNT = WS-AMOUNT * 0.05
   ELSE
      MOVE ZERO TO WS-DISCOUNT
   END-IF
END-IF. '''

from decimal import Decimal

ws_amount = Decimal('1500.00')
if ws_amount > Decimal('1000'):
    ws_discount = ws_amount * Decimal('0.10')
elif ws_amount > Decimal('500'):
    ws_discount = ws_amount * Decimal('0.05')
else:
    ws_discount = Decimal('0')

def test_discount():
    assert ws_discount == Decimal('0')


In [11]:
FastLanguageModel.for_inference(model)

test_prompt = """Translate this COBOL paragraph to modern Python. Use the 'decimal' module for all currency and math. Include a pytest assertion.

01 WS-TOTAL PIC S9(7)V99 VALUE 1000.00.
01 WS-COUNT PIC 9(3) VALUE 4.
01 WS-AVG PIC S9(7)V99 VALUE ZERO.
COMPUTE WS-AVG = WS-TOTAL / WS-COUNT."""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.1,
    top_p=0.9,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Translate this COBOL paragraph to modern Python. Use the 'decimal' module for all currency and math. Include a pytest assertion.

01 WS-TOTAL PIC S9(7)V99 VALUE 1000.00.
01 WS-COUNT PIC 9(3) VALUE 4.
01 WS-AVG PIC S9(7)V99 VALUE ZERO.
COMPUTE WS-AVG = WS-TOTAL / WS-COUNT. '''

from decimal import Decimal

ws_total = Decimal('1000.00')
ws_count = Decimal('4')
ws_avg = Decimal('250.00')

def test_seq():
    assert ws_avg == Decimal('250.00')


In [12]:
FastLanguageModel.for_inference(model)

test_prompt = """Translate this COBOL paragraph to modern Python. Use the 'decimal' module for all currency and math. Include a pytest assertion.

01 WS-TOTAL PIC S9(7)V99 VALUE ZERO.
01 WS-IDX PIC 9(3) VALUE 1.
PERFORM VARYING WS-IDX FROM 1 BY 1 UNTIL WS-IDX > 5
   ADD WS-IDX TO WS-TOTAL
END-PERFORM."""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.1,
    top_p=0.9,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Translate this COBOL paragraph to modern Python. Use the 'decimal' module for all currency and math. Include a pytest assertion.

01 WS-TOTAL PIC S9(7)V99 VALUE ZERO.
01 WS-IDX PIC 9(3) VALUE 1.
PERFORM VARYING WS-IDX FROM 1 BY 1 UNTIL WS-IDX > 5
   ADD WS-IDX TO WS-TOTAL
END-PERFORM. '''

from decimal import Decimal

ws_total = Decimal('0')
ws_idx = Decimal('1')
end_val = Decimal('5')
while ws_idx <= end_val:
    ws_total = ws_total + ws_idx
    ws_idx = ws_idx + Decimal('1')
